# Blindfolded Mini-Golf in the Wind & Robbins-Monro Algorithm

<img src="assets/blindfold-minigolf-logo.png" alt="Blindfolded Mini-Golf logo" width="450">

*You are playing a game of mini-golf — with a twist. You **cannot see the green**. Your job is to find how hard to putt using only your caddie's spoken feedback. Every attempt is a noisy hint, and with one simple rule — the **Robbins-Monro algorithm** — those hints add up to the perfect putting force.*

Welcome to the windiest blindfolded mini-golf course in the world!

You are standing on a mini-golf tee with a blindfold on. The green stretches out in front of you, but you **cannot see the hole** or where the ball ends up. Each turn, all you do is:

1. 🏌️ Choose a **putting force** (0 = barely a tap, 100 = full swing)
2. 🌬️ **Putt** — the wind nudges your ball by a random amount every single time
3. 📢 Listen to the caddie: *"3.2 meters PAST the hole"* or *"1.8 meters SHORT"*

The caddie always tells you **how far** you missed **and in which direction** — but never where the hole actually is.

### The four moving parts

Keep these four pieces in mind; everything below builds on them:

| | Piece | What it means |
|---|---|---|
| 🕶️ | **You're blindfolded** | You cannot see the hole or where the ball lands — only the caddie's words. |
| 🏌️ | **Force** | Your current *guess* of how hard to putt. This is the one number you control. |
| 🌬️ | **Wind** | A small random push added to every putt, so the ball never lands in exactly the same spot twice. |
| 📢 | **Signed feedback** | The caddie's report: a **positive** number means you went *past* the hole, a **negative** number means you fell *short*. |

### The challenge

Your goal: find the right putting force while blindfolded, using only the caddie's signed feedback over many putts.

This is the **Robbins-Monro problem** — finding an unknown target using noisy, one-at-a-time observations.

### What you'll learn

Through this game, you will discover:
- **Stochastic root-finding** — how to hit a target when you can only see *noisy* measurements of how far off you are
- **The Robbins-Monro algorithm** — the classic recipe for exactly this situation
- **Learning-rate decay** — why taking smaller and smaller steps is the secret to settling down
- **The Q-learning connection** — how this same pattern powers a cornerstone of reinforcement learning

Let's tee off — blindfold on. 🏌️

## Setup

Run the three cells below once before starting the exercises.

In [ ]:
# @title Install dependencies - run once {display-mode: "form"}
!pip install numpy matplotlib --quiet

In [ ]:
# @title Game utilities - run once {display-mode: "form"}
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Optional, Tuple


class MiniGolfGame:
    '''
    1D blindfolded mini-golf putting game with wind noise.

    You cannot see the hole. After each putt you receive a signed error:
        positive  = overshot  (ball went past the hole)
        negative  = undershot (ball stopped short)
    '''

    def __init__(self, seed=None, wind_std=8.0, drift_std=0.0):
        self.seed = seed if seed is not None else np.random.randint(0, 2**31)
        self.rng = np.random.RandomState(self.seed)
        self.wind_std = wind_std
        self.drift_std = drift_std
        self._force = 50.0
        self._hole_pos = self._init_hole()
        self.round = 0
        self.history = []

    def _init_hole(self):
        if self.rng.random() < 0.5:
            return 15.0 + self.rng.random() * 15.0
        else:
            return 65.0 + self.rng.random() * 15.0

    def _maybe_drift(self):
        if self.drift_std > 0:
            step = float(np.clip(self.rng.randn() * self.drift_std, -5.0, 5.0))
            self._hole_pos = float(np.clip(self._hole_pos + step, 10.0, 90.0))

    def putt(self, force):
        '''
        Putt with the given force. Wind randomly shifts the landing position.

        Returns a dict with:
            signed_error  - landing minus hole position
            landing       - where the ball ended up
            force         - the force used
            round         - current round number
        '''
        self.round += 1
        self._maybe_drift()
        wind = float(self.rng.randn() * self.wind_std)
        landing = float(np.clip(force + wind, 0.0, 100.0))
        signed_error = landing - self._hole_pos
        result = {
            'round': self.round,
            'force': float(force),
            'landing': landing,
            'signed_error': float(signed_error),
        }
        self.history.append(result)
        return result

    def reset(self):
        '''Reset to the initial state (same seed, same hole position).'''
        self.rng = np.random.RandomState(self.seed)
        self._force = 50.0
        self._hole_pos = self._init_hole()
        self.round = 0
        self.history = []

    def reveal_hole(self):
        '''Return the true hole position (for checking convergence after the exercise).'''
        return self._hole_pos

    @property
    def force(self):
        return self._force

    @force.setter
    def force(self, value):
        self._force = float(np.clip(value, 0.0, 100.0))


def plot_convergence(forces, signed_errors, learning_rates):
    '''Plot force convergence, signed errors, and learning-rate schedule.'''
    window = 10
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(forces, lw=2, color='#22c55e')
    axes[0].set_xlabel('Round')
    axes[0].set_ylabel('Putting force')
    axes[0].set_title('Force estimate over time')
    axes[0].grid(alpha=0.3)

    axes[1].plot(signed_errors, alpha=0.35, color='#f87171', label='Signed error')
    if len(signed_errors) >= window:
        rolling = np.convolve(signed_errors, np.ones(window) / window, mode='valid')
        axes[1].plot(range(window - 1, len(signed_errors)), rolling,
                     lw=2.5, color='#dc2626', label=f'{window}-round avg')
    axes[1].axhline(0, color='#22c55e', ls='--', lw=1.5, label='Target  (error = 0)')
    axes[1].set_xlabel('Round')
    axes[1].set_ylabel('Signed error')
    axes[1].set_title('Overshoot (+) / undershoot (−)')
    axes[1].legend(fontsize=9)
    axes[1].grid(alpha=0.3)

    axes[2].plot(learning_rates, lw=2, color='#60a5fa')
    axes[2].set_xlabel('Round')
    axes[2].set_ylabel('α')
    axes[2].set_title('Learning rate schedule')
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


game = MiniGolfGame(seed=42)
print('Blindfolded Mini-Golf utilities loaded!')
print('Blindfold on. Ready to find the hole from noisy feedback. \U0001f32c\ufe0f')

## Try the windy green

Before writing any code, play a few rounds yourself — it makes everything that follows click.

You are blindfolded, so resist peeking: don't press **Reveal hole** at first. Use the **+1 / −1** buttons (or the ← → arrow keys) to adjust your putting force, then press **PUTT** (or the Space bar). The caddie will tell you how far past or short you landed.

Try to zero in on the hole using only that feedback. How many putts does it take? The **Reveal hole** button is there for checking *after* you've played blind — to see how close your instincts got you.

In [ ]:
# @title Play Blindfolded Mini-Golf in the Wind - run to launch {display-mode: "form"}
# The tiny game is embedded directly below (base64), so this notebook is fully
# self-contained: it runs with no external files and no internet connection.
from IPython.display import HTML as _HTML

_GAME_B64 = "..."

_HTML(
    f'<iframe src="data:text/html;base64,{_GAME_B64}" '
    'width="100%" height="700" style="border:none; border-radius:12px;" '
    'title="Blindfolded Mini-Golf in the Wind"></iframe>'
)

### The score you actually get

Here's the key mind-shift. Unlike ordinary mini-golf, your job is **not to get lucky on one putt**. The blindfold means you can't inspect the green, and the wind makes every putt noisy — use the same force twice and the ball lands in two different spots. So no single putt can be trusted.

Instead, imagine using the same force many, many times. Sometimes the wind pushes the ball forward, sometimes backward. Over many putts, those random pushes mostly cancel out, and an average emerges. We call that average the **expected landing** — intuitively, it sits *somewhere in the middle* of where the wind has been scattering your ball.

Your real target is the force whose expected landing sits exactly on the hole:

$$
\mathbb{E}[\text{landing}] = \text{hole}
$$

(Read $\mathbb{E}[\cdots]$ as *"the long-run average of."*) And since the caddie reports the **error** — how far off you landed —

$$
\text{error} = \text{landing} - \text{hole},
$$

finding that force is the same as driving the *average* error to zero:

$$
\mathbb{E}[\text{error}] = 0
$$

That's the Robbins-Monro problem in one line: find the input where the average noisy feedback becomes zero.

A handy way to remember it:

> **Robbins-Monro doesn't chase the last putt. It chases the average putt.**

### This is root-finding with noise

Let's name things so we can talk about them precisely. There **is** a correct putting force in this game — but remember the catch:

- You are **blindfolded**, so you can't see the hole or where the ball lands
- You **don't know how far** the hole is
- After each putt you only hear a **signed error**: how far past or short you went
- The **wind** shifts the ball randomly every time

We'll write $\theta$ ("theta") for your putting force, and $\theta^*$ ("theta-star") for the ideal force — the one that lands the ball at the hole *on average*. Now define a function that measures, for any force, how far past the hole you land on average:

$$H(\theta) = \mathbb{E}[\text{landing} - \text{hole}] = \theta - \theta^*$$

In words: $H(\theta)$ is positive when you're putting too hard, negative when too soft, and exactly zero at the perfect force. So finding $\theta^*$ means finding the force where $H(\theta^*) = 0$. Mathematicians call the input that makes a function zero its **root** — so this is a **root-finding problem**.

The twist that makes it hard: you never get to see $H(\theta)$ cleanly. Each putt only gives you one **noisy sample** of it, because the wind is different every time.

### The same idea, picture version

If the math symbols feel abstract, here's the whole thing in plain language. You're looking for the force where the caddie's feedback **flips from SHORT to PAST**:

- Force **too small** → ball lands short → signed error is **negative**
- Force **too large** → ball lands past → signed error is **positive**
- The **right force** sits right at the changeover, where the *expected* signed error is **zero** — that's the root

Because of the wind, you'll almost never hear exactly *"0.0 meters"* on any single putt. But once you've found the right force, the caddie's feedback will **average out to zero** over many putts. That averaging-out is the whole trick.

### The Robbins-Monro putting rule

So how do you actually *find* that force? Robbins-Monro is a wonderfully simple **rule for adjusting your force after each blindfolded putt**. The gut-level version is just:

> **Overshot? Ease off. Fell short? Hit a bit harder.**

Here's the rule, step by step:

**① Putt with your current force.**
&nbsp;&nbsp;&nbsp;&nbsp;You start somewhere — say, force = 50.

**② Listen to the caddie's signed feedback.**
&nbsp;&nbsp;&nbsp;&nbsp;You get a noisy measurement of how far off you are — **and, crucially, in which direction.**

**③ Nudge your force against the error.**
&nbsp;&nbsp;&nbsp;&nbsp;Overshot (+)? Ease off a little. Undershot (−)? Hit a bit harder. In symbols:

$$\theta_{t+1} = \theta_t - \alpha_t \cdot \underbrace{(\text{landing}_t - \text{hole})}_{\text{noisy signed error } e_t}$$

&nbsp;&nbsp;&nbsp;&nbsp;The little $\alpha_t$ ("alpha") is your **step size** — how big an adjustment you make. The minus sign is what turns "overshot" into "ease off."

Repeat for many rounds and your force homes in on $\theta^*$.

**Why does this work?** Each signed error $e_t$ is a noisy peek at $H(\theta_t) = \theta_t - \theta^*$ — it leans the right way even though the wind muddies it. Subtracting it (scaled by $\alpha_t$) nudges you toward the target. As long as your steps are big enough to actually reach the root, yet small enough that the wind noise averages out, you're guaranteed to converge. (More on getting that balance right shortly.)

## Implement the Robbins-Monro Algorithm

Now it's your turn to teach the computer to play blindfolded. You'll write the Robbins-Monro rule as a short loop: putt, listen, adjust — over and over.

### Version 1: a fixed step size

We'll start with the simplest version, where the step size $\alpha$ stays the same every round. For each round $t = 0, 1, 2, \ldots$ you do three things:

1. **Read your current force** — your latest guess $\theta_t$, which the game keeps in `game.force`
2. **Putt** — call `game.putt(θ_t)`, which returns a result dictionary containing `signed_error`, the caddie's feedback $e_t = \text{landing}_t - \text{hole}$
3. **Adjust** using the Robbins-Monro rule:
   $$\theta_{t+1} = \theta_t - \alpha \cdot e_t$$

> 💭 *Sanity check before you code:* if $e_t = +5$ (you overshot by 5), should your new force be **higher** or **lower**? Trace it through the formula and make sure it matches your gut.

In [ ]:
def robbins_monro(
    game: MiniGolfGame,
    n_rounds: int = 100,
) -> Tuple[List[float], List[float], List[float]]:
    '''
    Play Blindfolded Mini-Golf with the Robbins-Monro rule and a fixed step size.

    Each round: read the current force, putt, listen to the caddie, and nudge
    the force against the signed error.

    Args:
        game:     MiniGolfGame instance
        n_rounds: how many putts to take

    Returns:
        Tuple of (forces, signed_errors, learning_rates)
    '''
    forces        = []
    signed_errors = []
    learning_rates = []

    game.reset()

    for t in range(n_rounds):
        # Fixed step size: the same boldness every round
        alpha_t = 3.0
        learning_rates.append(alpha_t)

        # -- 1. Read your current force (your latest guess) -------------------
        current_force = ...
        # --- 🎯🎯🎯🎯 ---
        forces.append(current_force)

        # -- 2. Putt, and hear the caddie's feedback -------------------------
        result = ...
        # --- 🎯🎯🎯🎯 ---
        signed_errors.append(result['signed_error'])

        # -- 3. Read the signed error: positive = past, negative = short -----
        e_t = ...
        # --- 🎯🎯🎯🎯 ---

        # -- 4. Robbins-Monro nudge: less force after past, more after short -
        # --- 🎯🎯🎯🎯 ---

    return forces, signed_errors, learning_rates

In [ ]:
forces, signed_errors, learning_rates = robbins_monro(game=game, n_rounds=100)

print(f'Starting force:       {forces[0]:.2f}')
print(f'Final force:          {forces[-1]:.2f}')
print(f'True hole position:   {game.reveal_hole():.2f}')
print(f'Final signed error:   {signed_errors[-1]:.2f} m')
print(f'Avg |error| last 10:  {np.mean(np.abs(signed_errors[-10:])):.2f} m')

In [ ]:
plot_convergence(forces, signed_errors, learning_rates)

## One crucial fix

Run the version above and you'll notice the force never quite *settles* — it keeps bouncing around the hole. Let's fix that.

### Why shrink the learning rate?

With a *fixed* step size, the force estimate keeps jittering around the hole and never fully settles. The reason is simple: even when you're already nearly perfect, the wind throws in a fake error, and your fixed-size step dutifully knocks you off the bullseye again. Big steps near the target are the problem.

The fix is to **shrink the step size as you go**. Be bold early, cautious late:

$$\alpha_t = \frac{\alpha_0}{1 + t / \text{decay_rate}}$$

- $\alpha_0$ — your **starting** boldness (how big your first adjustments are)
- $\text{decay_rate}$ — how *quickly* the steps shrink (bigger = slower shrink)

Why it works:
- **Early rounds:** big steps → race toward the right zone quickly
- **Later rounds:** small steps → fine-tune without overshooting
- **Eventually:** steps so tiny the estimate barely moves → it settles down

There's a beautiful balance hidden here, known as the **Robbins-Monro conditions**. Putting the plain-language version first:

- the steps must add up to *enough total travel* to reach the target no matter where you start — written $\sum_t \alpha_t = \infty$
- but their squares must add up to something *finite*, so the wind noise washes out — written $\sum_t \alpha_t^2 < \infty$

The decaying schedule above is the classic recipe that satisfies both at once.

In [ ]:
def robbins_monro(
    game: MiniGolfGame,
    n_rounds: int = 100,
    alpha_0: float = 5.0,
    decay_rate: float = 20.0,
) -> Tuple[List[float], List[float], List[float]]:
    '''
    Robbins-Monro with a shrinking step size: bold early, cautious late.

    Same putt-listen-adjust loop as before, but now the step size decays each
    round so the force can finally settle on the hole.

    Args:
        game:       MiniGolfGame instance
        n_rounds:   how many putts to take
        alpha_0:    starting step size (how bold the first adjustments are)
        decay_rate: how fast the step size shrinks (bigger = slower shrink)

    Returns:
        Tuple of (forces, signed_errors, learning_rates)
    '''
    forces         = []
    signed_errors  = []
    learning_rates = []

    game.reset()

    for t in range(n_rounds):
        # -- Shrinking step size: big early, tiny later ----------------------
        # --- 🎯🎯🎯🎯 ---
        learning_rates.append(alpha_t)

        # -- 1. Read your current force (your latest guess) -------------------
        # --- 🎯🎯🎯🎯 ---
        forces.append(current_force)

        # -- 2. Putt, and hear the caddie's feedback -------------------------
        # --- 🎯🎯🎯🎯 ---
        signed_errors.append(result['signed_error'])

        # -- 3. Read the signed error: positive = past, negative = short -----
        # --- 🎯🎯🎯🎯 ---

        # -- 4. Robbins-Monro nudge: less force after past, more after short -
        # --- 🎯🎯🎯🎯 ---

    return forces, signed_errors, learning_rates

In [ ]:
forces, signed_errors, learning_rates = robbins_monro(
    game=game,
    n_rounds=100,
    alpha_0=5.0,
    decay_rate=20.0,
)

print(f'Starting force:       {forces[0]:.2f}')
print(f'Final force:          {forces[-1]:.2f}')
print(f'True hole position:   {game.reveal_hole():.2f}')
print(f'Final signed error:   {signed_errors[-1]:.2f} m')
print(f'Avg |error| last 10:  {np.mean(np.abs(signed_errors[-10:])):.2f} m')

In [ ]:
plot_convergence(forces, signed_errors, learning_rates)

## The Q-learning connection

Here's the payoff. The rule you just built isn't a mini-golf trick — it's the exact shape of **Q-learning**, one of the foundational algorithms in reinforcement learning. Let's get there in four small steps, changing one thing at a time.

---

### Step 1 — Rewrite your putting rule as "move toward the target"

Your update was $\theta_{t+1} = \theta_t - \alpha_t \cdot e_t$, where the signed error was $e_t = \text{landing} - \text{hole}$. With a little rearranging, that's the same as:

$$\theta \;\leftarrow\; \theta + \alpha \cdot (\,\text{target} - \theta\,)$$

In plain words: **keep your old guess, and nudge it a fraction $\alpha$ of the way toward the target you just sampled.** In mini-golf, $\theta$ is your force and the "target" is the force that sinks the ball — each putt is a noisy hint about which way it lies.

---

### Step 2 — Let the single number be a *value* instead of a force

In reinforcement learning, an agent keeps an estimate called $Q(s, a)$ — read it as *"how good is it to take action $a$ when I'm in situation $s$?"* Pick **one** state-action pair and $Q(s,a)$ is just **one number you're trying to pin down** — exactly like your force $\theta$. Same shape, new label:

$$Q(s,a) \;\leftarrow\; Q(s,a) + \alpha \cdot (\,\text{target} - Q(s,a)\,)$$

---

### Step 3 — Fill in what the target should be

For a force, the target was "the force that hits the hole." For a value, the target comes from the famous **Bellman** idea: a good action is worth *the reward you get right now, plus the best you can do from wherever you land next.*

$$\text{target} \;=\; r + \gamma \max_{a'} Q(s', a')$$

Here $r$ is the immediate reward, $s'$ is the next situation, and $\gamma$ ("gamma") just says how much future rewards count.

---

### Step 4 — Put it together

Drop that target into the Step 2 rule and you get the **Q-learning update**:

$$Q(s,a) \;\leftarrow\; Q(s,a) + \alpha\,\big[\,\underbrace{r + \gamma \max_{a'} Q(s', a')}_{\text{target}} - Q(s,a)\,\big]$$

That bracket is just a **signed error** again — "how far my current value is from the freshly sampled target" — the value-world twin of the caddie's feedback.

---

### Where's the wind?

In mini-golf the **wind** was the source of randomness and the **blindfold** stopped you from peeking — so each putt was a *noisy* sample of the truth, not the truth itself.

In Q-learning, the "wind" is the **single sampled transition**. You don't get the true average target; you get *one* roll of the dice: one next-state $s'$, one reward $r$, scored against a $Q$ table that is itself still being learned. So every update is a noisy nudge — and the shrinking learning rate is exactly what averages that noise away, just like on the green.

---

> 💭 **One to mull over:** at the perfect force, the caddie's *expected* feedback was zero. By the same logic, what is the *expected* value of that Bellman bracket once $Q$ has fully converged? (Hint: a settled estimate doesn't want to move.)

## You found the root 🏌️

Look at what you just did. Mini-golf handed you a **stochastic root-finding** problem — hit an unseen target using only noisy feedback. **Robbins-Monro** gave you the update rule, and a **shrinking step size** let it settle. And **Q-learning** turns out to be the very same idea, with the caddie's feedback swapped for a Bellman error. One rule, blindfold and all — and now it's yours.